# Predicting Apartment Prices in Mexico City 🇲🇽

In [ ]:
# Import libraries here
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from glob import glob
from category_encoders import OneHotEncoder
from ipywidgets import Dropdown, FloatSlider, IntSlider, interact
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge  # noqa F401
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import make_pipeline


**Task 2.5.1: Write a wrangle function that takes the name of a CSV file as input and returns a DataFrame. The function should do the following steps:

Subset the data in the CSV file and return only apartments in Mexico City ("Distrito Federal") that cost less than $100,000.
Remove outliers by trimming the bottom and top 10% of properties in terms of "surface_covered_in_m2".
Create separate "lat" and "lon" columns.
Mexico City is divided into 15 boroughs. Create a "borough" feature from the "place_with_parent_names" column.
Drop columns that are more than 50% null values.
Drop columns containing low- or high-cardinality categorical values.
Drop any columns that would constitute leakage for the target "price_aprox_usd".
Drop any columns that would create issues of multicollinearity.

In [ ]:
# Build your `wrangle` function

def wrangle(filepath):
    # Read CSV file
    df = pd.read_csv(filepath)

In [ ]:
def wrangle(filepath):
    df = pd.read_csv(filepath)

    # Subset data: Apartments in "Distrito Federal", less than 400,000
    mask_ba = df["place_with_parent_names"].str.contains("Distrito Federal")
    mask_apt = df["property_type"] == "apartment"
    mask_price = df["price_aprox_usd"] < 100000
    df = df[mask_ba & mask_apt & mask_price]
    
    # Subset data: Remove outliers for "surface_covered_in_m2"
    low, high = df["surface_covered_in_m2"].quantile([0.1, 0.9])
    mask_area = df["surface_covered_in_m2"].between(low, high)
    df = df[mask_area]
    
    # Split "lat-lon" column
    df[["lat", "lon"]] = df["lat-lon"].str.split(",", expand=True).astype(float)
    df.drop(columns="lat-lon", inplace=True)
    
    # Create borough from place_with_parent_names
    df["borough"] = df["place_with_parent_names"].str.split("|", expand=True)[1]
    df.drop(columns="place_with_parent_names", inplace=True)
    
    # drop features with high null counts
    df.drop(columns = ['surface_total_in_m2', 'price_usd_per_m2', 'floor',
                      'rooms', 'expenses'], inplace = True)
    
    # Drop low-and high-cardinality categorical variables
    df.drop(columns=['operation', 'property_type', 'currency', 'properati_url'], inplace =True)
    
    # Drop Leaky
    df.drop(columns=['price',
    'price_aprox_local_currency',
   'price_per_m2'], inplace = True)
    
    # Drop column with multicollinearity
    ## No multicollinearity

    return df

In [ ]:
df = wrangle("data/mexico-city-real-estate-1.csv")
df.shape  # Should be (1101, 5)

In [ ]:
## to get missing values greater than 50%
df.isnull().sum()/len(df)

In [ ]:
## low and high cardinality
df.select_dtypes('object').nunique()

In [ ]:
## Check multicollinearity
corr = df.select_dtypes("number").drop(columns="price_aprox_usd").corr()
sns.heatmap(corr)

*** Task 2.5.2: Use glob to create the list files. It should contain the filenames of all the Mexico City 
real estate CSVs in the ./data directory, except for mexico-city-test-features.csv.

In [ ]:
files = glob("data/mexico-city-real-estate-*.csv")
files

### Combine your wrangle function, a list comprehension, and pd.concat to create a DataFrame df. It should contain all 
the properties from the five CSVs in files.

In [ ]:
frames = [wrangle(file) for file in files]
df = pd.concat(frames, ignore_index = True)
print(df.info())
df.head()

### Explore

In [ ]:
# Build histogram
plt.hist(df["price_aprox_usd"])

# Label axes
plt.xlabel("Price [$]")
plt.ylabel("Count")

# Add title
plt.title("Distribution of Apartment Prices")


In [ ]:
# Build scatter plot
plt.scatter(df["surface_covered_in_m2"], df["price_aprox_usd"])

# Label axes
plt.xlabel("Area [sq meters]")
plt.ylabel("Price [USD]")

# Add title
plt.title("Mexico City: Price vs. Area")

### Split
Task 2.5.7: Create your feature matrix X_train and target vector y_train. Your target is "price_aprox_usd". 
Your features should be all the columns that remain in the DataFrame you cleaned above.

In [ ]:
# Split data into feature matrix `X_train` and target vector `y_train`.

target = "price_aprox_usd"
features = df.drop(columns="price_aprox_usd")
y_train = df[target]
X_train = features

# Build Model

### Baseline
Calculate the baseline mean absolute error for your model.

In [ ]:
y_mean = y_train.mean()
y_pred_baseline = [y_mean] * len(y_train)
baseline_mae = mean_absolute_error(y_train, y_pred_baseline)
print("Mean apt price:", round(y_mean, 2))
print("Baseline MAE:", round(baseline_mae, 2))

### Iterate
Task 2.5.9: Create a pipeline named model that contains all the transformers necessary for this dataset and one of the 
predictors you've used during this project. Then fit your model to the training data.


In [ ]:
from category_encoders.one_hot import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline

# Step 1: Fill missing values in X_train manually (optional if using imputer)
X_train = X_train.copy()
X_train["surface_covered_in_m2"] = X_train["surface_covered_in_m2"].fillna(X_train["surface_covered_in_m2"].mean())
X_train["lat"] = X_train["lat"].fillna(X_train["lat"].mean())
X_train["lon"] = X_train["lon"].fillna(X_train["lon"].mean())
X_train["borough"] = X_train["borough"].fillna(X_train["borough"].mode()[0])

# Build 3-step model pipeline
model = make_pipeline(
    OneHotEncoder(use_cat_names=True),  # Step 1: Encoding
    SimpleImputer(),                    # Step 2: Imputation
    Ridge()                             # Step 3: Regression
)

# Fit model
model.fit(X_train, y_train)
from category_encoders.one_hot import OneHotEnco

## Evaluate
Read the CSV file mexico-city-test-features.csv into the DataFrame X_test

In [ ]:
X_test = pd.read_csv("data/mexico-city-test-features.csv")
print(X_test.info())
X_test.head()

### Use your model to generate a Series of predictions for X_test. When you submit your predictions to the grader, 
it will calculate the mean absolute error for your model.

In [ ]:
y_test_pred = pd.Series(model.predict(X_test))
y_test_pred.head()

## Communicate Results
Task 2.5.12: Create a Series named feat_imp. The index should contain the names of all the features your model 
considers when making predictions; the values should be the coefficient values associated with each feature. 
The Series should be sorted ascending by absolute value.


In [ ]:
coefficients = model.named_steps["ridge"].coef_
features = model.named_steps['onehotencoder'].get_feature_names()
feat_imp = pd.Series(coefficients, index=features)
feat_imp

### Create a horizontal bar chart that shows the 10 most influential coefficients for your model
Be sure to label your x- and y-axis "Importance [USD]" and "Feature", respectively, and give your chart 
the title "Feature Importances for Apartment Price". Use pandas.

In [ ]:
# Build bar chart
feat_imp.sort_values(key=abs).tail(10).plot(kind="barh")


# Label axes
plt.xlabel("Importance [USD]")
plt.ylabel("Feature")

# Add title
plt.title("Feature Importance for Apartment Price")
